In [0]:
use catalog dbw_kroger_insights;

In [0]:
WITH product_totals AS (
    SELECT
        product_id,
        SUM(views)     AS total_views,
        SUM(carts)     AS total_carts,
        SUM(purchases) AS total_purchases,
        SUM(revenue)   AS total_revenue
    FROM gold.ec_fact_product_daily
    GROUP BY product_id
),
banded_products AS (
    SELECT
        pt.*,
        CASE
            WHEN p.median_price IS NULL  THEN '8. Unpriced'
            WHEN p.median_price < 25     THEN '1. Under 25'
            WHEN p.median_price < 50     THEN '2. 25-50'
            WHEN p.median_price < 100    THEN '3. 50-100'
            WHEN p.median_price < 250    THEN '4. 100-250'
            WHEN p.median_price < 500    THEN '5. 250-500'
            WHEN p.median_price < 1000   THEN '6. 500-1000'
            ELSE                              '7. 1000+'
        END AS price_band_usd
    FROM product_totals pt
    LEFT JOIN gold.ec_dim_product p
        ON pt.product_id = p.product_id
)
SELECT
    price_band_usd,
    COUNT(*)                                                                    AS sku_count,
    SUM(total_views)                                                            AS total_views,
    SUM(total_purchases)                                                        AS total_purchases,
    ROUND(SUM(total_revenue), 2)                                                AS total_revenue,
    ROUND(100.0 * SUM(total_revenue) / SUM(SUM(total_revenue)) OVER (), 2)      AS revenue_share_pct,
    ROUND(100.0 * SUM(total_carts)     / NULLIF(SUM(total_views), 0), 2)        AS view_to_cart_pct,
    ROUND(100.0 * SUM(total_purchases) / NULLIF(SUM(total_views), 0), 2)        AS view_to_purchase_pct,
    ROUND(SUM(total_revenue) / NULLIF(SUM(total_views), 0), 2)                  AS revenue_per_view
FROM banded_products
GROUP BY price_band_usd
ORDER BY price_band_usd;

In [0]:
WITH product_conv AS (    
    SELECT        
        pt.product_id,        
        p.median_price,        
        CAST(pt.total_purchases AS DOUBLE) / NULLIF(pt.total_views, 0) AS conversion    
    FROM ( 
        SELECT 
            product_id,
            SUM(views) AS total_views,
            SUM(purchases) AS total_purchases
        FROM dbw_kroger_insights.gold.ec_fact_product_daily
        GROUP BY product_id
    ) pt    
    JOIN dbw_kroger_insights.gold.ec_dim_product p ON pt.product_id = p.product_id    
    WHERE pt.total_views >= 1000      
      AND p.median_price IS NOT NULL
      AND p.median_price > 0
),
ranked AS (    
    SELECT        
        *,        
        PERCENT_RANK() OVER (ORDER BY median_price) AS price_rank,        
        PERCENT_RANK() OVER (ORDER BY conversion)   AS conversion_rank    
    FROM product_conv
)
SELECT    
    COUNT(*)                                           AS products,    
    ROUND(corr(median_price, conversion), 3)           AS pearson_price,    
    ROUND(corr(ln(median_price), conversion), 3)       AS pearson_log_price,    
    ROUND(corr(price_rank, conversion_rank), 3)        AS spearman
FROM ranked;

In [0]:
WITH product_conv AS (    
    SELECT        
        pt.product_id,        
        p.category_l2,
        p.median_price,        
        CAST(pt.total_purchases AS DOUBLE) / NULLIF(pt.total_views, 0) AS conversion    
    FROM ( 
        SELECT 
            product_id,
            SUM(views) AS total_views,
            SUM(purchases) AS total_purchases
        FROM dbw_kroger_insights.gold.ec_fact_product_daily
        GROUP BY product_id
    ) pt    
    JOIN dbw_kroger_insights.gold.ec_dim_product p ON pt.product_id = p.product_id    
    WHERE pt.total_views >= 1000      
      AND p.median_price IS NOT NULL
      AND p.median_price > 0
      AND p.category_l2 NOT IN ('unknown', 'n/a')
),
ranked AS (    
    SELECT        
        product_id,
        category_l2,
        median_price,
        conversion,
        PERCENT_RANK() OVER (PARTITION BY category_l2 ORDER BY median_price) AS price_rank,        
        PERCENT_RANK() OVER (PARTITION BY category_l2 ORDER BY conversion)   AS conversion_rank    
    FROM product_conv
)
SELECT    
    category_l2,
    COUNT(*)                                           AS products,    
    ROUND(corr(price_rank, conversion_rank), 3)        AS spearman_corr
FROM ranked
GROUP BY category_l2
HAVING COUNT(*) >= 50
ORDER BY spearman_corr ASC;